# 02 · Candidate precursors
**Brief: Task 2 ("Brainstorm candidate precursors").** Each candidate has *a story* (why it could physically come
before a failure) and *a detector* (a simple yes/no rule). Code: `src/detectors.py` §4 (rules) and §5 (candidates).

### What this notebook answers from the brief
| Brief asks for | Where |
|---|---|
| **Task 2:** brainstorm candidates that might appear before an event | section 2 (10 ideas) |
| For each: the story and the detector | section 2 |
| A candidate can use one signal or combine several | single (C01–C04, C06, C07), combined (C05, C08–C10) |
| Use whatever helps you generate ideas: domain intuition, an LLM, an ML model | section 1 |

In [1]:
# --- Setup ----------------------------------------------------------------------------
# Libraries, plus the project's helper files in ../src (data.py, detectors.py, evaluation.py).
# Every analysis function this notebook calls lives there, in numbered sections (§1 ... §9).
import sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore", message="IProgress not found")  # harmless: kagglehub's progress bar

import pandas as pd
import plotly.io as pio

# Two outputs per chart: Plotly's own MIME type (rendered natively by VS Code and JupyterLab, works
# offline) plus an HTML fallback for classic Jupyter (loads plotly.js from a CDN).
pio.renderers.default = "plotly_mimetype+notebook_connected"
pd.set_option("display.max_colwidth", 120, "display.width", 200)  # show long rule texts in full

ROOT = Path.cwd().parent            # project root (this notebook lives in notebooks/)
sys.path.insert(0, str(ROOT / "src"))  # so `from data import ...` finds ../src/data.py

In [2]:
# --- Code used in this notebook (from ../src) --------------------------------------------
from data import (                            # src/data.py
    Config,                                   #   §1 settings: H, W, event, percentiles, seed
    load_panel, dev_holdout_split,            #   §2 raw data; which machines are dev / holdout
)
from detectors import (                       # src/detectors.py
    FeatureBank,                              #   §4 features: rolling means, error counts, CUSUM (cached)
    build_concepts,                           #   §5 the 10 candidate ideas and all their variants
)
from evaluation import (                      # src/evaluation.py
    learn_mapping, MAPPING,                   #   §7 which signal / error codes belong to which part (dev only)
    save_results,                             #   §9 save this notebook's key results for notebook 06
)

## 1 · Where the ideas came from
* **Domain intuition:** bearing wear raises vibration, friction slows rotation, a blockage raises pressure, an
  electrical fault disturbs voltage. Parts wear out with time since replacement, and controllers log error codes.
* **The data:** notebook 01 found one signature signal per part, plus many look-alike drifts that an error code
  tells apart. That suggested pairing a signal with an error code.
* **Part pairing and part age:** since each part has its own signal and its own error code, C09 requires both
  for the *same* part. And a drift on a part long past its last replacement is more likely to be real, which
  led to C10 (drift on an aging part).

**Conclusion.** Four families of ideas: one signal drifting, an error code, a part overdue for replacement, and
combinations of these.

## 2 · The candidates
> **Relies on:** `src/data.py` §1–2 (`Config`, `load_panel`, `dev_holdout_split`) · `src/detectors.py` §4–5
> (`FeatureBank`, `build_concepts`)

In [3]:
panel = load_panel()
bank = FeatureBank(panel)            # computes features (rolling means, error counts, ...) once and caches them
cfg = Config(event="any")            # <- change H, W or the event ("any", "comp1".."comp4") here

# Split MACHINES (not hours) 50/50 into dev and holdout, stratified by machine model.
# Every choice below is made on dev; holdout is only used to report. Splitting by machine means
# no machine helps choose a detector and then also grades it.
dev, hold = dev_holdout_split(panel, cfg.dev_fraction, cfg.seed)

# C09 ("matched signature") and C10 need to know which signal and which error codes belong to which part.
# That pairing is learned on DEV machines only, so holdout labels never leak into it.
MAPPING.clear()
MAPPING.update(learn_mapping(panel, cfg, dev))
print("dev-learned pairing:", {c: (v["signal"], v["errors"]) for c, v in MAPPING.items()})

# Every candidate idea, with its story, first-draft rule and all the variants the sweep will try.
concepts = build_concepts(cfg.percentiles, cfg.event, MAPPING)
print(len(concepts), "candidate ideas,", sum(len(c.variants) for c in concepts), "variants in total")

dev-learned pairing: {'comp1': ('volt', ['error1']), 'comp2': ('rotate', ['error2', 'error3']), 'comp3': ('pressure', ['error4']), 'comp4': ('vibration', ['error5'])}
10 candidate ideas, 655 variants in total


In [4]:
# Every candidate written out: its family, story and first-draft rule (the rule before any tuning).
for c in concepts:
    print(f"{c.cid} · {c.title}   [{c.family}; {len(c.variants)} variants to test]")
    print(f"   story:    {c.story}")
    print(f"   detector: {c.default.rule_text()}\n")

C01 · Voltage creeps up   [telemetry; 95 variants to test]
   story:    An electrical part that is degrading (e.g. a winding or contactor) disturbs supply regulation, so the measured voltage drifts above its usual band before it gives out.
   detector: volt hourly reading > P99 of normal operation (all machines) for 3+ consecutive hours

C02 · Rotation sags   [telemetry; 95 variants to test]
   story:    Wear or a seizing part adds friction and drag, so the shaft turns slower than usual for the same drive in the days before a mechanical failure.
   detector: rotate hourly reading < P1 of normal operation (all machines) for 3+ consecutive hours

C03 · Pressure builds   [telemetry; 95 variants to test]
   story:    A sticking valve, clogged passage or failing seal restricts flow, so upstream pressure rises before the component fails.
   detector: pressure hourly reading > P99 of normal operation (all machines) for 3+ consecutive hours

C04 · Vibration rises   [telemetry; 95 variants to t

**Conclusion.** Ten candidates:

| ID | Candidate | Family | Story in one line |
|---|---|---|---|
| C01 | Voltage creeps up | telemetry | a degrading electrical part disturbs the supply |
| C02 | Rotation sags | telemetry | wear adds friction, so the shaft slows |
| C03 | Pressure builds | telemetry | a sticking valve or blocked passage restricts flow |
| C04 | Vibration rises | telemetry | bearing wear or a loose mount shakes the machine |
| C05 | Any signal drifts | combo | one alarm watching all four signals at once |
| C06 | Any error code | errors | controllers log faults when a part misbehaves |
| C07 | Overdue maintenance | counter | the longer since replacement, the closer to failure |
| C08 | Drift + any error | combo | a drift confirmed by an error within two days |
| C09 | Matched signature | combo | a part's **own** signal drifts **and** its **own** error code is logged |
| C10 | Drift on an aging part | combo | a drift on an old part is likelier to be real; warns before the error code |

## 3 · How each idea becomes many variants
Notebook 04 tests every candidate in many **variants**, so ideas are compared fairly, not on a lucky first draft:
* **smoothing:** the raw reading, held for 3+ hours, a 6 h or 24 h trailing mean, or CUSUM;
* **threshold scope:** fleet-wide, per model, per machine, or robust per machine (median + z·MAD);
* **percentile** of normal operation: P95 to P99.99.

**Conclusion.** 655 variants in total, every threshold set from normal-operation hours only, as the brief
requires.

## 4 · Save this notebook's results
The candidate list is saved to `outputs/02_candidates.json` for notebook 06.

In [5]:
save_results("02_candidates", dict(                          # src/evaluation.py §9
    candidates=[dict(id=c.cid, title=c.title, family=c.family, story=c.story,
                     first_draft_rule=c.default.rule_text(), variants=len(c.variants)) for c in concepts],
    total_variants=sum(len(c.variants) for c in concepts),
), cfg)

saved outputs/02_candidates.json


PosixPath('/Users/avijay/Documents/oden_precursors/outputs/02_candidates.json')